# Bài học 22: Mở rộng sản phẩm

## Thêm agent mới vào team

Trong bài học cuối cùng này, chúng ta sẽ đi qua một kịch bản thực tế: **thêm agent thứ 4 vào team** — agent hiệu đính kiểm tra ngữ pháp, độ dễ đọc, và SEO best practice.

Chúng ta sẽ lần theo quy trình phát triển từ Bài 15 (Tìm hiểu → Lên kế hoạch → Triển khai → Kiểm chứng → Lặp lại), và bạn sẽ **kiểm chứng kết quả** bằng kiến thức đã học.

Đây là cách bạn sẽ làm việc trong thực tế:
1. Bạn mô tả điều mình muốn
2. Claude Code triển khai
3. Bạn kiểm chứng kết quả bằng kiến thức của mình

## Bước 1: Tìm hiểu

Trước khi thực hiện bất kỳ thay đổi nào, Claude Code sẽ khám phá codebase. Đây là prompt bạn sẽ đưa ra:

> "Tôi muốn thêm agent hiệu đính vào team. Trước khi thay đổi gì, hãy khám phá codebase và giải thích: (1) các agent được định nghĩa như thế nào, (2) chúng được thêm vào team ra sao, (3) cần tuân theo pattern tool nào."

Claude Code sẽ đọc:
- `output/backend/agents/content_writer.py` — để xem pattern định nghĩa agent
- `output/backend/agents/team.py` — để xem cách các thành viên được lắp ráp
- `output/backend/tools/storage.py` — để xem pattern tool

Hãy tự làm tương tự — đọc các file quan trọng:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Đọc một agent hiện có để hiểu pattern
writer_path = os.path.abspath("../../output/backend/agents/content_writer.py")
with open(writer_path, "r", encoding="utf-8") as f:
    print("=== agents/content_writer.py ===")
    print(f.read())

print("\n")

team_path = os.path.abspath("../../output/backend/agents/team.py")
with open(team_path, "r", encoding="utf-8") as f:
    print("=== agents/team.py ===")
    print(f.read())

## Bước 2: Lên kế hoạch

Sau khi hiểu codebase, bạn sẽ nói với Claude Code:

> "Lên kế hoạch thêm agent hiệu đính. Agent cần đọc bài viết, kiểm tra ngữ pháp/độ dễ đọc/SEO, sửa lỗi, và lưu bài viết đã cập nhật. Dùng Claude Sonnet với structured output cho báo cáo hiệu đính."

Kế hoạch tốt sẽ bao gồm:
1. **File agent mới** `agents/proofreader.py`: với tools `get_article_content` và `update_article_content` + `output_schema` cho báo cáo hiệu đính
2. **Schema** định nghĩa ngay trong file agent (giống các agent khác — không cần file schema riêng)
3. **Thêm vào team** trong `agents/team.py`: thêm `proofreader` vào danh sách members
4. **Cập nhật `__init__.py`**: re-export agent mới

Hãy xây dựng và kiểm chứng từng phần:

## Bước 3: Triển khai — Schema

Việc đầu tiên là định nghĩa schema cho structured output. Tuân theo pattern Pydantic từ Bài học 11:

In [ ]:
from pydantic import BaseModel, Field

class ProofreadIssue(BaseModel):
    issue_type: str = Field(description="Type: grammar, readability, seo, factual")
    location: str = Field(description="Which section has the issue")
    description: str = Field(description="What the issue is")
    suggestion: str = Field(description="How to fix it")
    severity: str = Field(default="medium", description="low, medium, or high")

class ProofreadResult(BaseModel):
    corrected_article: str = Field(description="Full article with corrections applied")
    issues_found: list[ProofreadIssue] = Field(description="List of issues found and fixed")
    readability_score: int = Field(description="Readability score 1-10")
    seo_score: int = Field(description="SEO optimization score 1-10")
    word_count: int = Field(description="Word count of corrected article")

# Kiểm tra schema hoạt động (không gọi API)
test = ProofreadResult(
    corrected_article="# Test\n\nCorrected article.",
    issues_found=[
        ProofreadIssue(
            issue_type="grammar", location="Paragraph 2",
            description="Run-on sentence", suggestion="Split into two sentences",
        ),
        ProofreadIssue(
            issue_type="seo", location="Introduction",
            description="Target keyword missing", suggestion="Add keyword to first paragraph",
            severity="high",
        ),
    ],
    readability_score=7, seo_score=8, word_count=150,
)

print("Validation schema thành công!")
print(f"Số vấn đề: {len(test.issues_found)}")
for issue in test.issues_found:
    print(f"  [{issue.severity.upper()}] {issue.issue_type}: {issue.description}")

## Bước 3: Triển khai — Agent

Claude Code sẽ tạo `agents/proofreader.py` tuân theo pattern hiện có:

In [ ]:
# Đây là code mà agents/proofreader.py sẽ trông như
proofreader_code = '''
"""Proofreader -- kiểm tra ngữ pháp, độ dễ đọc, SEO best practice."""

from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.storage import get_article_content, update_article_content

proofreader = Agent(
    name="Proofreader",
    role="Check and fix grammar, readability, and SEO issues in articles",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=[get_article_content, update_article_content],
    instructions=[
        "You proofread SEO articles for grammar, readability, and SEO best practices.",
        "Use get_article_content to read the article.",
        "Check for: grammar errors, awkward phrasing, readability issues,"
        " keyword usage, SEO best practices (title, headings, meta).",
        "Fix all issues directly in the article text.",
        "Use update_article_content to save the corrected article.",
        "Report what you found and fixed.",
        "Never use emojis or icons.",
    ],
    markdown=True,
)
'''

print("Định nghĩa agent mà Claude Code sẽ tạo:")
print("=" * 50)
print(proofreader_code)

## Bước 3: Triển khai — Thêm vào team

Trong `agents/team.py`, thêm thành viên mới:

```python
from .proofreader import proofreader

members = [content_writer, aio_analyzer, proofreader]  # Đã thêm!
if image_finder is not None:
    members.append(image_finder)
```

Và thêm vào instructions của team leader:
```python
"- Proofreader: checking grammar, readability, SEO best practices in articles",
```

Vậy là xong. Team leader giờ đã biết về proofreader và sẽ giao việc hiệu đính cho agent này.

## Bước 4: Kiểm chứng

Đây là lúc kiến thức CỦA BẠN phát huy tác dụng. Kiểm chứng phần triển khai:

In [ ]:
checks = [
    ("Bài 5", "Agent có vấn đề về knowledge cutoff không?",
     "Không -- hiệu đính chỉ cần nội dung bài viết, không cần thông tin real-time. Không cần search tools."),
    ("Bài 6", "Instructions có được cấu trúc tốt không?",
     "Có -- Vai trò ('proofreader'), Nhiệm vụ ('check grammar, SEO'), Tools ('get_article_content, update_article_content')."),
    ("Bài 7", "Lựa chọn model có đúng không?",
     "\u0110úng -- Claude Sonnet. Hỗ trợ tools. Giỏi phân tích và sửa văn bản."),
    ("Bài 9", "Các tools có phù hợp không?",
     "Có -- get_article_content để đọc, update_article_content để lưu. Cùng tools mà Image Finder dùng."),
    ("Bài 17", "Tích hợp team có hoạt động không?",
     "Có -- chỉ cần thêm vào danh sách members và cập nhật instructions của leader. TeamMode.tasks xử lý giao việc."),
    ("Bài 18", "Storage có xử lý được cập nhật không?",
     "Có -- update_article_content là thread-safe. Số từ được tự động tính lại."),
    ("Bài 19", "Kiến trúc có vẫn hợp lý không?",
     "Có -- một file mới (agents/proofreader.py), một dòng trong team.py. Kiến trúc phẳng được duy trì."),
]

print("=== Danh sách kiểm chứng ===\n")
for module, question, answer in checks:
    print(f"  [{module}]")
    print(f"  H: {question}")
    print(f"  \u0110: {answer}")
    print()

## Vibecoding frontend

Giao diện React frontend cũng được xây dựng với Claude Code. Cách làm:

**Prompt:**
> "Tôi có FastAPI backend tại localhost:7777 với các endpoint: POST /teams/seo-workspace/runs (SSE streaming chat), GET /api/articles (danh sách), GET /api/articles/{id} (chi tiết), DELETE /api/articles/{id}. Xây dựng React + Vite frontend với: giao diện chat stream phản hồi, sidebar danh sách bài viết, và trình xem bài viết render Markdown."

**Claude Code tạo ra:**
- `output/frontend/package.json` — dependencies (react, react-markdown, vite)
- `output/frontend/vite.config.js` — dev server với proxy tới backend
- `output/frontend/src/api.js` — fetch wrapper với SSE streaming
- `output/frontend/src/components/Chat.jsx` — giao diện chat streaming
- `output/frontend/src/components/ArticleList.jsx` — sidebar với article card
- `output/frontend/src/components/ArticleView.jsx` — trình render bài viết đầy đủ

**Điểm mấu chốt:** Bạn không cần biết React. Bạn mô tả **hành vi** mình muốn ("stream phản hồi", "danh sách bài viết ở sidebar", "render Markdown") và Claude Code xử lý phần triển khai.

**Cải tiến từng bước:**
1. Lần đầu: layout cơ bản và chat
2. "Thêm SSE streaming để phản hồi hiển thị từng từ một"
3. "Thêm sidebar bài viết tự động cập nhật"
4. "Thêm nút xóa trên mỗi article card"

Mỗi lần cải tiến xây dựng trên lần trước. Đây chính là **vibecoding** — mô tả điều bạn muốn, kiểm chứng, lặp lại.

## Thêm ý tưởng mở rộng

Những thứ bạn có thể yêu cầu Claude Code xây dựng:

| Mở rộng | Độ khó | Prompt cho Claude Code |
|---------|--------|------------------------|
| Thay đổi giọng văn | Dễ | "Cập nhật instructions của Content Writer để viết theo phong cách thân thiện, gần gũi" |
| Thêm kiểm tra mật độ từ khóa | Trung bình | "Thêm tool đếm tần suất xuất hiện từ khóa trong bài viết" |
| Thêm agent dịch thuật | Trung bình | "Thêm thành viên thứ 4 dịch bài viết sang tiếng Việt bằng Claude Sonnet" |
| Thêm Google Search Console | Khó | "Tạo GSC toolkit trong tools/ tuân theo pattern của DataForSEOSearchTools" |
| Thêm xác thực người dùng | Khó | "Thêm JWT authentication vào serve.py dùng AgentOS middleware" |

Mỗi mở rộng đều tuân theo cùng quy trình: **Tìm hiểu -> Lên kế hoạch -> Triển khai -> Kiểm chứng**.

## Chu trình phát triển

```
  Ý tưởng     "Tôi muốn thêm hiệu đính"
    |
    v
  Kế hoạch   Claude Code khám phá, bạn duyệt phương án
    |
    v
  Viết code  Claude Code viết code
    |
    v
  Kiểm chứng BẠN kiểm tra bằng kiến thức của mình
    |
    v
  Sử dụng   Chạy thử, dùng thật, lặp lại
    |
    +-------> (quay lại Ý tưởng cho lần cải tiến tiếp theo)
```

**Bạn là người kiểm soát chất lượng. Claude Code là người xây dựng. Kiến thức của bạn là thứ giúp bạn làm việc hiệu quả.**

Nếu không có kiến thức từ khóa học này, bạn không thể kiểm chứng lựa chọn model có đúng không, tools có phù hợp không, hay tích hợp team có chính xác không. Giờ thì bạn có thể.

## Bạn đã hoàn thành tất cả 22 bài học.

Toàn bộ hành trình:

### Mô-đun 1: Nền tảng Python (Bài học 1-4)
Biến, list, dictionary, hàm, package — nền tảng để đọc hiểu code.

### Mô-đun 2: Hiểu về AI (Bài học 5-7)
Cách LLM hoạt động, prompt và ngữ cảnh, lựa chọn model — lý do đằng sau mọi quyết định.

### Mô-đun 3: Xây dựng Agent (Bài học 8-12)
Agent đầu tiên, tools, structured output, chuỗi agent, lưu trữ — thực hành xây dựng agent.

### Mô-đun 4: Phát triển với AI (Bài học 13-18)
Bước ngoặt tư duy, sử dụng Claude Code, prompting/kế hoạch/học hỏi, Content Writer, Supporting Agents, Team — từ kỹ năng đến sản phẩm.

### Mô-đun 5: Sản phẩm hoàn chỉnh (Bài học 19-22)
Cách mọi thứ kết nối, kiến thức web cơ bản, giao diện web, mở rộng với vibecoding.

---

### Tiếp theo là gì?

1. **Bắt đầu sử dụng sản phẩm** — chạy `python output/backend/serve.py` và tạo bài viết
2. **Tùy chỉnh instructions của agent** — điều chỉnh phong cách viết, giọng văn, chiến lược SEO
3. **Thử Claude Code** — thực hiện thay đổi nhỏ (ví dụ: đổi giọng văn của writer)
4. **Xây dựng tools riêng** — thêm tools mà team SEO cần
5. **Dạy người khác** — giờ bạn đã hiểu đủ để hướng dẫn đồng nghiệp

**Cách tốt nhất để học là xây dựng. Bắt đầu nhỏ, kiểm chứng cẩn thận, lặp lại nhanh.**

In [ ]:
# Cuối cùng: khám phá toàn bộ cấu trúc dự án
output_dir = os.path.abspath("../../output/backend")
print("Codebase production của bạn:\n")

for root, dirs, files in os.walk(output_dir):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(output_dir, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = len(f.readlines())
            print(f"{sub_indent}{file} ({lines} dòng)")

print("\nBạn hiểu rõ từng file trong số này.")
print("Kiến thức đó là siêu năng lực của bạn khi làm việc với Claude Code.")